# TOX3GNN – Optuna + 30-Run  Experiment
 **T4 GPU**

## 1 · Mount Google Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Edit this to match where your TOX folder lives in Drive ──────
DRIVE_ROOT = '/content/drive/MyDrive/AUA/Thesis/code/HybridGNN/TOX'
os.makedirs(f'{DRIVE_ROOT}/checkpoints_tox21', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results_tox21',     exist_ok=True)

print('Drive mounted. Root:', DRIVE_ROOT)
print('Contents:', os.listdir(DRIVE_ROOT))

Mounted at /content/drive
Drive mounted. Root: /content/drive/MyDrive/AUA/Thesis/code/HybridGNN/TOX
Contents: ['tox21_dataset (1).csv', 'tox21_dataset.csv', 'TOX3GNN.py', 'optim_TOX3GNN.py', 'results_tox21', 'feat_corr.py', '__pycache__', 'imputation_checkpoints', 'tox21_imputed.csv', 'checkpoints_tox21', 'tox21_analysis.ipynb', 'TOX3GNN_experiment.ipynb', 'utils.py', 'TOX3GNN_multitask.ipynb']


## 2 · Install dependencies

In [7]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'optuna'],  check=True)

import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}')
print('Dependencies ready.')


torch=2.11.0+cu128  cuda=True
Dependencies ready.


## 3 · Imports

In [8]:
import os, sys, time, random, shutil, warnings, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import optuna

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, GATConv, GINConv, SAGEConv
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from rdkit import Chem

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


Device: cuda


## 4 · Load `utils.py` from Drive
This imports your **exact** implementations — nothing is rewritten here.

In [9]:
# utils.py must be in DRIVE_ROOT alongside tox21_dataset.csv
sys.path.insert(0, DRIVE_ROOT)

from utils import (
    one_hot_encoding,
    get_atom_features,
    get_bond_features,
    smiles_to_graph_list,
    scaffold_split,
    save_ckp,
    load_ckp,
    optimizer_to,
    round_to_4,
)

# Alias used throughout this notebook



print('utils.py loaded from Drive — using your exact implementations.')


utils.py loaded from Drive — using your exact implementations.


## 5 · Configuration

In [10]:
# ── Paths ────────────────────────────────────────────────────────────────────
# Pass either the imputed or original dataset .csv path
DATA_CSV  = f'{DRIVE_ROOT}/tox21_dataset.csv'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints_tox21'
RESULTS_DIR    = f'{DRIVE_ROOT}/results_tox21'

# ── Optuna toggle ─────────────────────────────────────────────────────────────
# Set RUN_OPTUNA = True to search for hyperparameters first.
RUN_OPTUNA   = False
N_TRIALS     = 20      # number of Optuna trials (each trains 50 epochs)

# Change to False if you are running on the full dataset
IS_DROPPING = True
# ── Known-best params (used when RUN_OPTUNA = False, or as Optuna fallback) ──
KNOWN_LAYER_TYPES = ['sage', 'gin', 'sage']
KNOWN_HIDDEN = 400
KNOWN_DROPOUT = 0.25
KNOWN_LR = 0.0001
KNOWN_WD = 1e-4
# ── 30-run experiment settings ────────────────────────────────────────────────
N_RUNS = 10
MAX_EPOCHS = 500
EVAL_EVERY = 10
PATIENCE = 20
NUM_GRAPHS_PER_BATCH = 64
USE_SCAFFOLD_SPLIT = True
GLOBAL_SEED = 43
# set it to your chosen task
TASK_NAME = 'SR-ARE'

print('Config OK')
print(f'  RUN_OPTUNA={RUN_OPTUNA}  N_TRIALS={N_TRIALS}')
print(f'  scaffold_split={USE_SCAFFOLD_SPLIT}  N_RUNS={N_RUNS}  MAX_EPOCHS={MAX_EPOCHS}')


Config OK
  RUN_OPTUNA=False  N_TRIALS=20
  scaffold_split=True  N_RUNS=10  MAX_EPOCHS=500


## 6 · Load dataset & build splits

In [11]:
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

# Read the imputed tox21 or the original tox21 data
df_raw= pd.read_csv(DATA_CSV)
print(f'Size of the Dataset: {len(df_raw)}\n')
# All task columns — everything except smiles/mol_id
task_cols = [col for col in df_raw.columns if col not in ['smiles', 'mol_id']]
NUM_TASKS = len(task_cols)
print(f'Tasks ({NUM_TASKS}): {task_cols}')

df_raw[['smiles', 'mol_id']] = df_raw[['smiles', 'mol_id']].astype('str')
df_raw[task_cols] = df_raw[task_cols].apply(pd.to_numeric, errors = 'coerce').astype('float32')

# Dtype casting
from utils import get_valid_mask

def prep_data(df_source:pd.DataFrame, IS_DROPPING:bool, target_task:str = TASK_NAME):
  """
  Data preperation: Drops the null values (IS_DROPPING = True)
  and uses validation mask to filter out invalid SMILES
  Parameters:
  df_source: pd.DataFrame
  IS_DROPPING: bool
  target_task: str
  """
  print(f'Prepping the Tox21 Task {TASK_NAME} Data...')
  df_task = df_source[['smiles', target_task]].copy()
  if IS_DROPPING:
    print('Dropping the null values...')
    df_task = df_task.dropna(subset=[target_task]).reset_index(drop=True)
    print(f'Dropped {len(df_task)} rows')
  else:
    print('Continuing without Dropping the NUlls')

  print("\nCreating the SMILES validation mask...")
  smiles_valid_mask = get_valid_mask(df_task['smiles'])

  df_clean = df_task[smiles_valid_mask].reset_index(drop=True)
  print(f'Valid SMILES: {len(df_clean)} / {len(df_task)}')

  X_smiles = df_clean['smiles'].tolist()
  y_labels = df_clean[[target_task]].values.astype(np.float32)

# Multi-task extraction of X, y
  # X_smiles, y_labels = [], []
  # for _, row in df_clean.iterrows():
  #   smi = row['smiles']
  #   X_smiles.append(smi)
  #   y_labels.append(row[target_task])
  # y_labels = np.array(y_labels)
  print(f'Label matrix shape: {y_labels.shape}')

  print('Building molecular graphs...')
  data_list = smiles_to_graph_list(X_smiles, y_labels)
  print(f'Graph list: {len(data_list)} molecules  |  y shape per graph: {data_list[0].y.shape}')
  # assert len(data_list) == len(X_smiles)

  return data_list, X_smiles, y_labels

# data_list, X_smiles, y_labels = prep_data(df_raw,IS_DROPPING, TASK_NAME)


from rdkit.Chem.rdmolops import GetAdjacencyMatrix
import torch
from torch_geometric.data import Data

def split_data(data_list, X_smiles, train_frac:float, val_frac:float,
               y_labels, USE_SCAFFOLD_SPLIT:bool=True):
  """
   Splits the data: scaffold split (USE_SCAFFOLD_SPLIT=True) or a random split(USE_SCAFFOLD_SPLIT=False)
  Parameters:

  data_list: pd.DataFrame
  X_smiles
  """
  if USE_SCAFFOLD_SPLIT:
      train_idx, val_idx, test_idx = scaffold_split(X_smiles, train_frac=train_frac,
                                                    val_frac=val_frac, seed=GLOBAL_SEED)
  else:
      all_idx = list(range(len(data_list)))
      test_size = 1.0 - train_frac
      train_idx, temp = train_test_split(all_idx, test_size=test_size, random_state=GLOBAL_SEED)
      val_idx, test_idx = train_test_split(temp, test_size=0.5, random_state=GLOBAL_SEED)
      print(f'Random split → train:{len(train_idx)} val:{len(val_idx)} test:{len(test_idx)}')

  val_loader  = DataLoader([data_list[i] for i in val_idx],
                          batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False, drop_last=False)
  test_loader = DataLoader([data_list[i] for i in test_idx],
                          batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False, drop_last=False)
  trainval_data = [data_list[i] for i in list(train_idx) + list(val_idx)]
  train_data    = [data_list[i] for i in train_idx]

  print('Data splits ready.')
  return train_data, val_loader, test_loader, trainval_data, train_idx, val_idx, test_idx

# Version 2 to avoid probable data leakage

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

def prep_single_split(df_source: pd.DataFrame, IS_DROPPING: bool, target_task: str = TASK_NAME):
    df_task = df_source[['smiles', target_task]].copy()

    if IS_DROPPING:
        df_task = df_task.dropna(subset=[target_task]).reset_index(drop=True)

    smiles_valid_mask = get_valid_mask(df_task['smiles'])
    df_clean = df_task[smiles_valid_mask].reset_index(drop=True)

    X_smiles = df_clean['smiles'].tolist()
    y_labels = df_clean[[target_task]].values.astype(np.float32)

    data_list = smiles_to_graph_list(X_smiles, y_labels)
    return data_list, X_smiles, y_labels


def split_raw_data(df_raw: pd.DataFrame, train_frac: float, val_frac: float, USE_SCAFFOLD_SPLIT: bool = True):
    smiles_list = df_raw['smiles'].tolist()

    if USE_SCAFFOLD_SPLIT:
        train_idx, val_idx, test_idx = scaffold_split(
            smiles_list, train_frac=train_frac, val_frac=val_frac, seed=GLOBAL_SEED
        )
    else:
        all_idx = list(range(len(df_raw)))
        test_size = 1.0 - train_frac
        train_idx, temp_idx = train_test_split(all_idx, test_size=test_size, random_state=GLOBAL_SEED)

        # Calculate relative split for remaining validation and test sets
        val_relative_ratio = val_frac / test_size
        val_idx, test_idx = train_test_split(temp_idx, train_size=val_relative_ratio, random_state=GLOBAL_SEED)

    df_train = df_raw.iloc[train_idx].reset_index(drop=True)
    df_val   = df_raw.iloc[val_idx].reset_index(drop=True)
    df_test  = df_raw.iloc[test_idx].reset_index(drop=True)
    df_trainval = pd.concat([df_train, df_val], ignore_index=True)

    return df_train, df_val, df_test, df_trainval

TRAIN_FRAC = 0.8
VAL_FRAC = (1.0 - TRAIN_FRAC) / 2.0
# Scaffold or Random splits
df_train, df_val, df_test, df_trainval = split_raw_data(
    df_raw, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, USE_SCAFFOLD_SPLIT=USE_SCAFFOLD_SPLIT
)

train_data, X_train, y_train = prep_single_split(df_train, IS_DROPPING, TASK_NAME)
val_data,   X_val,   y_val   = prep_single_split(df_val,   IS_DROPPING, TASK_NAME)
test_data,  X_test,  y_test  = prep_single_split(df_test,  IS_DROPPING, TASK_NAME)
trainval_data, _, _          = prep_single_split(df_trainval, IS_DROPPING, TASK_NAME)
# DataLoaders
train_loader = DataLoader(train_data, batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False)

Size of the Dataset: 7831

Tasks (12): ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


[12:04:30] WARNING: not removing hydrogen atom without neighbors
[12:04:30] Explicit valence for atom # 8 Al, 6, is greater than permitted
[12:04:30] Explicit valence for atom # 3 Al, 6, is greater than permitted
[12:04:30] Explicit valence for atom # 4 Al, 6, is greater than permitted
[12:04:31] Explicit valence for atom # 4 Al, 6, is greater than permitted
[12:04:32] Explicit valence for atom # 9 Al, 6, is greater than permitted
[12:04:32] Explicit valence for atom # 5 Al, 6, is greater than permitted
[12:04:32] Explicit valence for atom # 16 Al, 6, is greater than permitted
[12:04:33] Explicit valence for atom # 20 Al, 6, is greater than permitted


Scaffold split → train: 6264 (79.99%), val: 783 (10.00%), test: 784 (10.01%)


[12:04:34] Explicit valence for atom # 8 Al, 6, is greater than permitted
[12:04:34] Explicit valence for atom # 3 Al, 6, is greater than permitted
[12:04:34] Explicit valence for atom # 4 Al, 6, is greater than permitted
[12:04:34] Explicit valence for atom # 4 Al, 6, is greater than permitted
[12:04:34] Explicit valence for atom # 9 Al, 6, is greater than permitted
[12:04:34] Explicit valence for atom # 5 Al, 6, is greater than permitted
[12:04:34] Explicit valence for atom # 16 Al, 6, is greater than permitted
[12:04:53] Explicit valence for atom # 8 Al, 6, is greater than permitted
[12:04:53] Explicit valence for atom # 3 Al, 6, is greater than permitted
[12:04:53] Explicit valence for atom # 4 Al, 6, is greater than permitted
[12:04:53] Explicit valence for atom # 4 Al, 6, is greater than permitted
[12:04:53] Explicit valence for atom # 9 Al, 6, is greater than permitted
[12:04:53] Explicit valence for atom # 5 Al, 6, is greater than permitted
[12:04:53] Explicit valence for atom 

## 7 · GNN model

In [12]:
class GNN(torch.nn.Module):
    def __init__(self, layer_types, hidden_dim, dropout, n_tasks=1):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        in_dim = 79
        for lt in layer_types:
            if lt == 'gcn':
                self.convs.append(GCNConv(in_dim, hidden_dim))
            elif lt == 'gat':
                self.convs.append(GATConv(in_dim, hidden_dim))
            elif lt == 'gin':
                mlp = nn.Sequential(
                    nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
                    nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
                self.convs.append(GINConv(mlp, eps=0.00005, train_eps=True))
            elif lt == 'sage':
                self.convs.append(SAGEConv(in_dim, hidden_dim))
            else:
                raise ValueError(f'Unknown layer type: {lt}')
            in_dim = hidden_dim
        self.drop = nn.Dropout(p=dropout)
        # n_tasks outputs — one logit per task
        self.out  = Linear(hidden_dim * 2, n_tasks)

    def forward(self, x, edge_index, batch_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = torch.tanh(x)
        x = self.drop(x)
        x = torch.cat([gmp(x, batch_index), gap(x, batch_index)], dim=1)
        return self.out(x)   # shape (batch_size, n_tasks)


def multitask_loss(logits, targets, pos_weight_tensor):
    """
    BCE loss over all tasks, masking out NaN labels.
    logits  : (batch, n_tasks)
    targets : (batch, n_tasks)  — may contain NaN
    pos_weight_tensor: (n_tasks,)
    """
    valid_mask = ~torch.isnan(targets)          # (batch, n_tasks) bool
    if valid_mask.sum() == 0:
        return torch.tensor(0.0, requires_grad=True, device=logits.device)
    # Replace NaN with 0 so BCE doesn't error — masked out anyway
    targets_clean = targets.clone()
    targets_clean[~valid_mask] = 0.0
    # Compute per-element BCE with per-task pos_weight
    criterion = torch.nn.BCEWithLogitsLoss(
        # -[w*y*sig(y)+w*(1-y)*sig(1-y)]
        pos_weight=pos_weight_tensor, reduction='none')
    loss_all = criterion(logits, targets_clean)  # (batch, n_tasks)
    # Zero out the NaN positions and average over valid only
    loss_all = loss_all * valid_mask.float()
    return loss_all.sum() / valid_mask.float().sum()


def multi_evaluate_auc(model, loader, device, task_cols):
    """
    Mean ROC-AUC across all tasks, skipping tasks where the loader
    has fewer than 2 unique labels (can't compute AUC).
    """
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x.float(), batch.edge_index, batch.batch)
            all_logits.append(logits.cpu().numpy())
            all_targets.append(batch.y.cpu().numpy())
    logits_np  = np.concatenate(all_logits,  axis=0)   # (N, n_tasks)
    targets_np = np.concatenate(all_targets, axis=0)   # (N, n_tasks)


    task_aucs = []
    for t_idx, t_name in enumerate(task_cols):
        y_true = targets_np[:, t_idx]
        y_score = torch.sigmoid(torch.tensor(logits_np[:, t_idx])).numpy()
        valid   = ~np.isnan(y_true)
        if valid.sum() < 2 or len(np.unique(y_true[valid])) < 2:
            continue   # skip tasks with no positive or no negative examples
        task_aucs.append(roc_auc_score(y_true[valid], y_score[valid]))

    return float(np.mean(task_aucs)) if task_aucs else float('nan')

def evaluate_auc(model, loader, device, task_cols):
    """
    Mean ROC-AUC across all tasks, skipping tasks where the loader
    has fewer than 2 unique labels (can't compute AUC).
    """
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x.float(), batch.edge_index, batch.batch)
            all_logits.append(logits.view(-1).cpu().numpy())
            all_targets.append(batch.y.view(-1).cpu().numpy())
    logits_np  = np.concatenate(all_logits,  axis=0)
    targets_np = np.concatenate(all_targets, axis=0)

    y_pred =torch.sigmoid(torch.tensor(logits_np)).numpy()
    y_true = targets_np

    valid = ~np.isnan(y_true)
    y_true_valid = y_true[valid]
    y_pred_valid = y_pred[valid]
    y_unique = np.unique(y_true_valid)
    if len(y_unique) < 2:
        print(f"Unique values in Target Y: {y_unique}")
        return float('nan')

    return float(roc_auc_score(y_true_valid, y_pred_valid))

print('GNN and helpers defined.')
print(f'Output layer: {len(task_cols) if "task_cols" in dir() else "n_tasks"} tasks')


GNN and helpers defined.
Output layer: 12 tasks


## 8 · Optuna hyperparameter search

In [13]:
# Positional Argument
# TODO devide to DEVICE
def train_target_pw_data_list(data_list, train_idx, device = device):
  train_targets = np.array([data_list[i].y.item() for i in train_idx])
  pos_count = np.sum(train_targets == 1)
  neg_count = np.sum(train_targets == 0)
  pos_weight = torch.tensor([neg_count / max(pos_count, 1)], dtype=torch.float32).to(device)
  return train_targets, pos_weight

def train_target_pw_graph(train_data, device = device):
  """
  Craetes a positional weight to compensate
  for the class imbalance in the loss function
  Parameters:
  train_data: pd.DataFrame
  device: torch.device
  Returns:
  train_targets: torch.tensor
  pos_weight_tensor: torch.tensor
  """
  train_targets = torch.tensor([graph.y.item() for graph in train_data if graph.y is not None])

  pos_count = np.sum(train_targets == 1)
  neg_count = np.sum(train_targets == 0)
  pos_weight = torch.tensor([neg_count / max(pos_count, 1)], dtype=torch.float32).to(device)
  return train_targets, pos_weight

def multi_task_pos_weight(data_list, train_idx, num_tasks = NUM_TASKS, device = device):
  train_targets = np.array([data_list[i].y.detach().cpu().reshape(-1).numpy() for i in train_idx])  # (N_train, n_tasks)
  print(f"train_targets shape: {train_targets.shape}")
  pw_list = []
  for col in range(num_tasks):
      col_vals  = train_targets[:, col]
      valid     = ~np.isnan(col_vals)
      pos_count = (col_vals[valid] == 1).sum()
      neg_count = (col_vals[valid] == 0).sum()
      weight    = neg_count / max(pos_count, 1) if pos_count > 0 else 1.0
      pw_list.append(weight)
      print(f'  {task_cols[col]:<15}: pos={pos_count}  neg={neg_count}  weight={weight:.2f}')

  pos_weight_tensor = torch.tensor(pw_list, dtype=torch.float32).to(device)
  print(f'\npos_weight_tensor shape: {pos_weight_tensor.shape}  (one per task)')
  return train_targets, pos_weight_tensor



In [14]:
# ── Per-task positive weights computed from training labels only ─────────────

# ── Optuna (single-task proxy on for speed; full multi-task too slow) ──

SRATE_IDX = task_cols.index('SR-ARE') if 'SR-ARE' in task_cols else 0

if RUN_OPTUNA:
    optuna_train_loader = DataLoader(
        [train_data[i] for i in train_idx],
        batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True, drop_last=True)

    def objective(trial):
        layer_types = [trial.suggest_categorical(f'layer_{i}', ['gcn','gat','gin','sage'])
                       for i in range(3)]
        hidden_dim  = trial.suggest_int('hidden_dim', 64, 256, step=32)
        dropout     = trial.suggest_float('dropout', 0.0, 0.5)
        lr     = trial.suggest_float('lr', 1e-5, 1e-3, log=True)

        m  = GNN(layer_types, hidden_dim, dropout, n_tasks=NUM_TASKS).to(device)
        opt  = torch.optim.Adam(m.parameters(), lr=lr)
        best_val, patience_ctr = 0.0, 0

        for epoch in range(100):
            m.train()
            for batch in optuna_train_loader:
                batch = batch.to(device)
                opt.zero_grad()
                logits = m(batch.x.float(), batch.edge_index, batch.batch)
                loss   = multitask_loss(logits, batch.y, pos_weight_tensor)
                loss.backward()
                opt.step()

            val_auc = evaluate_auc(m, val_loader, device, task_cols)
            trial.report(val_auc, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
            if val_auc > best_val:
                best_val, patience_ctr = val_auc, 0
            else:
                patience_ctr += 1
                if patience_ctr >= 5:
                    break
        return best_val

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    study  = optuna.create_study(direction='maximize', pruner=pruner)
    print(f'Running Optuna ({N_TRIALS} trials)...')
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best_p= study.best_trial.params
    best_layer_types = [best_p[f'layer_{i}'] for i in range(3)]
    best_hidden = best_p['hidden_dim']
    best_dropout     = round_to_4(best_p['dropout'])
    best_lr = round_to_4(best_p['lr'])
    print(f'Best AUC: {study.best_trial.value:.4f}  layers={best_layer_types}')

    from datetime import datetime
    optuna_path = os.path.join(RESULTS_DIR, f'optuna_results_{datetime.today().strftime("%Y-%m-%d")}.txt')
    with open(optuna_path, 'w') as f:
        f.write('Optuna Study Results\n' + '='*60 + '\n')
        f.write(f'Best AUC: {study.best_trial.value:.6f}\n')
        for k, v in study.best_trial.params.items():
            f.write(f'  {k}: {round_to_4(v)}\n')
        f.write('\nAll trials:\n')
        for rank, t in enumerate(sorted(
            [t for t in study.trials if t.value is not None],
            key=lambda t: t.value, reverse=True), 1):
            f.write(f'{rank}. Trial #{t.number}: AUC={t.value:.6f} | {t.params}\n')
    print(f'Optuna results saved to {optuna_path}')

else:
    best_layer_types = KNOWN_LAYER_TYPES
    best_hidden = KNOWN_HIDDEN
    best_dropout     = KNOWN_DROPOUT
    best_lr     = KNOWN_LR
    print(f'Using known-best params: {best_layer_types}  hidden={best_hidden}')


Using known-best params: ['sage', 'gin', 'sage']  hidden=400


## 9 · Save helpers

In [15]:
import csv
import os
import json

def append_task_run_result(results_dir, task_name, run_id, test_auc, best_val_auc, auc_history, train_loss_history):
    """Safely appends structured JSON arrays to a task-specific CSV without parser errors."""
    os.makedirs(results_dir, exist_ok=True)
    # Generates filenames like 'SR-ARE_runs_summary.csv'
    csv_filename = f"{task_name}_runs_summary.csv"
    csv_path = os.path.join(results_dir, csv_filename)
    file_exists = os.path.exists(csv_path)

    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['task', 'model', 'run', 'test_auc', 'best_val_auc', 'train_loss_history', 'val_auc_history'])
        writer.writerow([
            task_name, NAME, run_id, f'{test_auc:.6f}', f'{best_val_auc:.6f}',
            json.dumps(train_loss_history), json.dumps(auc_history)
        ])

## 10 · 30-Run experiment


In [16]:
# Set the Task Checkpoint
NAME = '_'.join(best_layer_types)
def make_task_ckp(CHECKPOINT_DIR, current_task, NAME):
    task_ckpt_dir  = os.path.join(CHECKPOINT_DIR, current_task, NAME)
    task_model_dir = os.path.join(CHECKPOINT_DIR, current_task, NAME + '_best')
    os.makedirs(task_ckpt_dir, exist_ok=True)
    os.makedirs(task_model_dir, exist_ok=True)
    return task_ckpt_dir, task_model_dir


In [ ]:
def train_target_pw_graph(train_data, device=device):
  """
  Craetes a positional weight to compensate
  for the class imbalance in the loss function
  Parameters:
  train_data: pd.DataFrame
  device: torch.device
  Returns:
  train_targets: torch.tensor
  pos_weight_tensor: torch.tensor
  """
  train_targets = torch.tensor([graph.y.item() for graph in train_data if graph.y is not None])

  # Fixed: Use torch.sum instead of np.sum for torch.Tensor
  pos_count = torch.sum(train_targets == 1)
  neg_count = torch.sum(train_targets == 0)
  # Fixed: Convert pos_count to a Python scalar using .item() for the max function
  pos_weight = torch.tensor([neg_count.item() / max(pos_count.item(), 1)], dtype=torch.float32).to(device)
  return train_targets, pos_weight

print(f'Model : {NAME}  |  Tasks: {TASK_NAME}  |  Runs: {N_RUNS}')
# mean-across-tasks test AUC per run
all_run_aucs = []
experiment_start = time.time()
train_targets, pos_weight_tensor = train_target_pw_graph(train_data, device=device)
task_ckpt_dir, task_model_dir = make_task_ckp(CHECKPOINT_DIR, TASK_NAME, NAME)
for run in range(N_RUNS):
    run_start    = time.time()


    run_seed = run
    torch.manual_seed(run_seed)
    np.random.seed(run_seed)
    random.seed(run_seed)

    model     = GNN(best_layer_types, best_hidden, best_dropout, n_tasks=1).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=best_lr, weight_decay=KNOWN_WD)

    train_loader = DataLoader(
        train_data,
        batch_size=NUM_GRAPHS_PER_BATCH,
        shuffle=True, drop_last=True,
        generator=torch.Generator().manual_seed(run_seed),
    )

    best_val_auc = 0.0
    patience_ctr = 0
    auc_history  = []
    train_loss_list = []

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=6,
    min_lr=1e-6
    )
    for epoch in range(MAX_EPOCHS):
        model.train()
        epoch_loss = 0.0
        num_batches = 0
        for batch in train_loader:
            batch  = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch.x.float(), batch.edge_index, batch.batch)
            # loss   = multitask_loss(logits, batch.y, pos_weight_tensor)
            loss = F.binary_cross_entropy_with_logits(logits.view(-1), batch.y.view(-1),pos_weight=pos_weight_tensor)
            loss.backward()
            optimizer.step()

            epoch_loss +=loss.item()
            num_batches +=1
        epoch_mean_loss = epoch_loss/num_batches
        train_loss_list.append(round(epoch_mean_loss, 6))


        if (epoch + 1) % EVAL_EVERY == 0:
            val_auc = evaluate_auc(model, val_loader, device, task_cols)
            auc_history.append(round(val_auc, 6))

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                patience_ctr = 0
                state = {
                    'epoch':      epoch + 1,
                    'state_dict': model.state_dict(),
                    'optimizer':  optimizer.state_dict(),
                    'val_auc':    best_val_auc,
                  'epoch_mean_loss': epoch_mean_loss,
                    'train_history': train_loss_list,
                    'auc_history': auc_history,
                    'layers':     best_layer_types,
                    'hidden':     best_hidden,
                    'dropout':    best_dropout,
                    'lr':         best_lr,
                }
                save_ckp(state, True, task_ckpt_dir, task_model_dir,
                             f'model_run{run:02d}.pt', f'best_model_run{run:02d}.pt')
            else:
                patience_ctr += 1

            if patience_ctr >= PATIENCE:
              # TODO: add the train_loss
                print(f'  Run {run:02d} | Early stop @ ep {epoch+1} | best Val AUC {best_val_auc:.4f}')
                break

    # Load best checkpoint for test evaluation
    best_ckpt = os.path.join(task_model_dir, f'best_model_run{run:02d}.pt')
    if os.path.exists(best_ckpt):
        model, optimizer, _ = load_ckp(best_ckpt, model, optimizer)
    test_auc = evaluate_auc(model, test_loader, device, task_cols)
    auc_history.append(round(test_auc, 6))
    all_run_aucs.append(test_auc)

    # Multi-task test AUC
    # test_multi_auc = multi_evaluate_auc(model, test_loader, device, task_cols)
    # all_run_aucs.append(test_multi_auc)
    # model.eval()
    # all_logits, all_targets = [], []
    # with torch.no_grad():
    #     for batch in test_loader:
    #         batch = batch.to(device)
    #         all_logits.append(model(batch.x.float(), batch.edge_index, batch.batch).cpu().numpy())
    #         all_targets.append(batch.y.cpu().numpy())
    # logits_np  = np.concatenate(all_logits,  axis=0)
    # targets_np = np.concatenate(all_targets, axis=0)

    # per_task_auc = {}
    # for t_idx, t_name in enumerate(task_cols):
    #     y_true  = targets_np[:, t_idx]
    #     y_score = torch.sigmoid(torch.tensor(logits_np[:, t_idx])).numpy()
    #     valid   = ~np.isnan(y_true)
    #     if valid.sum() >= 2 and len(np.unique(y_true[valid])) >= 2:
    #         per_task_auc[t_name] = round(roc_auc_score(y_true[valid], y_score[valid]), 4)
    #     else:
    #         per_task_auc[t_name] = float('nan')

    # mean_test_auc = float(np.nanmean(list(per_task_auc.values())))


    # all_run_aucs.append(mean_test_auc)

    run_time_sec = time.time() - run_start
    # all_run_times.append(run_time_sec)

    run_mins   = run_time_sec / 60
    total_mins = (time.time() - experiment_start) / 60

    # Append to CSV
    append_task_run_result(
        RESULTS_DIR, TASK_NAME, run, test_auc, best_val_auc, auc_history, train_loss_list
    )

    # Per-run progress printout
    print(f'Run {run+1:02d} | Task: {TASK_NAME:<12} | Test AUC: {test_auc:.4f} | Val AUC: {best_val_auc:.4f} | '
          f'Run Time: {run_mins:.2f}m | Total Time: {total_mins:.2f}m')


# ==============================================================================
# FINAL EXPERIMENT SUMMARY
# ==============================================================================
# total_exp_time = (time.time() - experiment_start) / 60

print(f'\n{"="*60}')
print(f'EXPERIMENT COMPLETE — Model: {NAME} | Task: {TASK_NAME}')
print(f'{"="*60}')
print(f'  Mean Test AUC : {np.mean(all_run_aucs):.4f} ± {np.std(all_run_aucs):.4f}')
print(f'  Min Test AUC  : {np.min(all_run_aucs):.4f}')
print(f'  Max Test AUC  : {np.max(all_run_aucs):.4f}')
print(f'------------------------------------------------------------')
# print(f'  Mean Time/Run : {np.mean(all_run_times)/60:.2f} m')
# print(f'  Total Duration: {total_exp_time:.2f} m')
print(f'  Saved Results : {RESULTS_DIR}/{TASK_NAME}_runs_summary.csv')
print(f'{"="*60}')

Model : sage_gin_sage  |  Tasks: SR-ARE  |  Runs: 10


In [ ]:
import os, glob, json

csv_files = glob.glob(os.path.join(RESULTS_DIR, '*_runs_summary.csv'))

if not csv_files:
    print(f"No summary files found in {RESULTS_DIR}. Run the experiment first!")
else:
    # Concatenate all task summaries into a single DataFrame for unified analysis
    df_list = [pd.read_csv(f) for f in csv_files]


In [ ]:
import os, glob, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Search for all generated summary CSVs matching the pattern *_runs_summary.csv
csv_files = glob.glob(os.path.join(RESULTS_DIR, '*_runs_summary.csv'))

if not csv_files:
    print(f"No summary files found in {RESULTS_DIR}. Run the experiment first!")
else:
    # Concatenate all task summaries into a single DataFrame for unified analysis
    df_list = [pd.read_csv(f) for f in csv_files]
    df_res = pd.concat(df_list, ignore_index=True)

    df_res['test_auc'] = df_res['test_auc'].astype(float)
    df_res['best_val_auc'] = df_res['best_val_auc'].astype(float)

    # 1. Overall Summary Table across Tasks
    print("==================================================================")
    print("                    PER-TASK PERFORMANCE SUMMARY                  ")
    print("==================================================================")
    summary_table = df_res.groupby('task')['test_auc'].agg(
        Mean='mean', Std='std', Median='median', Min='min', Max='max', Runs='count'
    ).reset_index()

    print(summary_table.to_string(index=False))

    overall_mean = df_res['test_auc'].mean()
    overall_std = df_res['test_auc'].std()
    print("-" * 66)
    print(f"OVERALL AVERAGE TEST AUC ACROSS ALL TASKS: {overall_mean:.4f} ± {overall_std:.4f}\n")

    # 2. Visualizations Dashboard
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    sns.set_theme(style="whitegrid")

    # Plot A: Boxplot of Test AUC by Task
    sns.boxplot(data=df_res, x='task', y='test_auc', ax=axes[0, 0], palette="Set2")
    axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45, ha='right')
    axes[0, 0].set_title("Test ROC-AUC Distribution per Task", fontsize=14, fontweight='bold')
    axes[0, 0].set_ylabel("ROC-AUC")

    # Plot B: Mean Performance Comparison (Ranked)
    ranked_summary = summary_table.sort_values(by='Mean', ascending=False)
    sns.barplot(data=ranked_summary, x='Mean', y='task', ax=axes[0, 1], palette="Blues_r")
    axes[0, 1].set_title("Ranked Average Test AUC per Task", fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel("Mean ROC-AUC")

    # Plot C: Validation Trajectories
    for idx, row in df_res.iterrows():
        try:
            val_hist = json.loads(row['val_auc_history'])
            axes[1, 0].plot(range(len(val_hist)), val_hist, alpha=0.15, color='gray')
        except:
            pass

    all_val_hists = [json.loads(h) for h in df_res['val_auc_history'].dropna()]
    max_len = max([len(h) for h in all_val_hists])
    padded_hists = [h + [np.nan]*(max_len - len(h)) for h in all_val_hists]
    mean_val_curve = np.nanmean(padded_hists, axis=0)

    axes[1, 0].plot(range(len(mean_val_curve)), mean_val_curve, color='red', linewidth=2.5, label='Mean Validation AUC')
    axes[1, 0].set_title("Validation AUC Trajectories Across All Runs & Tasks", fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel("Evaluation Step")
    axes[1, 0].set_ylabel("Validation ROC-AUC")
    axes[1, 0].legend()

    # Plot D: Training Loss Trajectories
    for idx, row in df_res.iterrows():
        try:
            loss_hist = json.loads(row['train_loss_history'])
            axes[1, 1].plot(range(len(loss_hist)), loss_hist, alpha=0.15, color='steelblue')
        except:
            pass

    axes[1, 1].set_title("Training Loss Trajectories Across All Runs & Tasks", fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("BCE Loss")

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'per_task_evaluation_dashboard.png'), dpi=300)
    plt.show()

## 11 · Statistical significance

> Add blockquote


Run after you have 30-run results for all 4 baselines. Paste the AUC lists below.

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats

# Ensure RESULTS_DIR and NAME match your notebook variables
csv_path = os.path.join(RESULTS_DIR, 'all_runs_summary.csv')

if os.path.exists(csv_path):
    df_results = pd.read_csv(csv_path)

    # Extract hybrid model scores using mod_auc
    hybrid_scores = df_results[df_results['model'] == NAME]['mod_auc'].values
    print(f'Hybrid ({NAME}): mean={np.median(hybrid_scores):.4f} ± {np.std(hybrid_scores):.4f}\n')

    header = f'{"Model":<12}  {"Mean":>7}  {"Std":>7}  {"p-value":>12}  {"Significant?":>13}'
    print(header)
    print('-' * len(header))

    models = df_results['model'].unique()
    for m in models:
        if m == NAME:
            continue
        baucs = df_results[df_results['model'] == m]['mod_auc'].values
        if len(baucs) < 2:
            continue
        _, p = stats.mannwhitneyu(hybrid_scores, baucs, alternative='greater')
        sig = 'YES p<0.05' if p < 0.05 else 'no'
        print(f'{m:<12}  {np.median(baucs):>7.4f}  {np.std(baucs):>7.4f}  {p:>12.2e}  {sig:>13}')
else:
    print(f"Summary CSV not found at {csv_path}. Run Cell 10 first.")

## 12 · Results plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot( y=all_run_aucs, ax=axes[0], color='steelblue', width=0.4)
sns.stripplot(y=all_run_aucs, ax=axes[0], color='navy', alpha=0.5, jitter=True)
axes[0].axhline(np.median(all_run_aucs), color='red', ls='--',
                label=f'median={np.median(all_run_aucs):.4f} ± {np.std(all_run_aucs):.4f}')
axes[0].set_title(f'{N_RUNS}-run mean-task AUC distribution\n{NAME}')
axes[0].set_ylabel('Mean ROC-AUC (all tasks)')
axes[0].legend()

summary_df  = pd.read_csv(os.path.join(RESULTS_DIR, 'all_runs_summary.csv'))
hybrid_rows = summary_df[summary_df['model'] == NAME]
# Column name written by append_run_result is 'auc_step_N'
auc_cols    = [c for c in summary_df.columns if c.startswith('auc_step_')]
for _, row in hybrid_rows.iterrows():
    axes[1].plot(range(len(auc_cols)), row[auc_cols].values, alpha=0.25, color='steelblue')
axes[1].set_xlabel(f'Eval checkpoint (every {EVAL_EVERY} epochs)')
axes[1].set_ylabel('Val mean ROC-AUC (all tasks)')
axes[1].set_title(f'Val AUC over training — all {N_RUNS} runs')

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, f'{NAME}_results.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved to {plot_path}')


## 13 · Load a saved checkpoint (optional)
Use this cell to reload the best model from any run — e.g. to run inference or inspect weights.
The checkpoint saved in Cell 10 contains the full model state including which layers were used.


In [ ]:
torch.serialization.add_safe_globals([np._core.multiarray.scalar])

# ── Edit run_id to load whichever run you want ────────────────────────────────
run_id  = 0
load_name    = NAME   # or hardcode e.g. 'sage_gin_sage'
load_model_dir = os.path.join(CHECKPOINT_DIR, load_name + '_best')
ckpt_path    = os.path.join(load_model_dir,
                            f'model_{load_name}_run{run_id:02d}_tox21.pt')

# Reconstruct the model with the same architecture
loaded_model = GNN(best_layer_types, best_hidden, best_dropout).to(device)
loaded_opt   = torch.optim.Adam(loaded_model.parameters(), lr=best_lr)

loaded_model, loaded_opt, start_epoch = load_ckp(ckpt_path, loaded_model, loaded_opt)

# Verify
auc = evaluate_auc(loaded_model, test_loader, device)
print(f'Loaded run {run_id} checkpoint (trained to epoch {start_epoch})')
print(f'Test AUC on reload: {auc:.4f}')